# MaxViT-Tiny (timm, ImageNet-1k, 224 px) — DIMER image classification and bounded fine-tuning (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/maxvit-classification-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/maxvit-classification-pipeline/blob/main/tutorials/maxvit_classification_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-timm%2Fmaxvit__tiny__tf__224.in1k-ffcc4d?style=flat)](https://huggingface.co/timm/maxvit_tiny_tf_224.in1k) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fmaxvit-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/maxvit) [![arXiv](https://img.shields.io/badge/arXiv-2204.01697-b31b1b.svg)](https://arxiv.org/abs/2204.01697) [![License](https://img.shields.io/badge/License-Apache--2.0-green.svg)](https://github.com/kurtvalcorza/maxvit-classification-pipeline/blob/main/LICENSE)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.1 — **standalone** (§4)  
**Capability:** ImageNet-1k image classification with MaxViT-Tiny, a hybrid convolution and multi-axis attention network, and a bounded fine-tune that replaces the 1000-class head with one for your own classes, compares it on a held-out split with the majority-class and zero-shot ImageNet baselines, and exports a reloadable SafeTensors adapter

**This notebook is standalone.** It carries the repository's package (2 modules under `src/maxvit_classification_pipeline/`, at revision `6b5938290319`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `041f2cce4d74c7539d63aa9fb85786e78072d487` (~124 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned checkpoint, downloads and digest-verifies the pinned CIFAR-10 subset, validates and splits it, measures the majority-class and zero-shot ImageNet baselines, probes the model with a blank and a noise image, **runs the bounded fine-tune**, evaluates the held-out split, classifies an unseen split, exports the adapter, reloads it onto a fresh base model to verify the predictions, and writes machine-readable outputs with provenance. Nothing is skipped behind a default-off flag, and no clone or DIMER worker is required (NOTEBOOK_SPEC 2.1 §5, RUN7, FT2).

**Bring Your Own Data:** Two optional BYOD branches are included, and both are off by default (`USE_BYOD_IMAGE = False`, `USE_BYOD_DATASET = False`). `USE_BYOD_IMAGE` runs your own image through the same validation, ImageNet prediction and evaluation-report stages as the sample. `USE_BYOD_DATASET` takes your own class folders through the full adaptation workflow — validate, split, baselines, fine-tune, evaluate, export and reload — under NOTEBOOK_SPEC 2.1 DAT14. Set `BYOD_IMAGE_PATH` or `BYOD_DATASET_PATH` to read from a location without an upload dialog (EXE2).

MaxViT (Multi-Axis Vision Transformer) is a hybrid network. Every block combines an MBConv convolution with two self-attention steps: **block attention** inside non-overlapping 7×7 windows, for local detail, and **grid attention** across a sparse 7×7 grid spanning the whole feature map, for global context. The Tiny model stacks such blocks in four stages behind a convolutional stem, about 30.9M parameters and 5.6 GMACs. The `timm/maxvit_tiny_tf_224.in1k` checkpoint was trained on ImageNet-1k by the paper authors in TensorFlow and ported to PyTorch in `timm`. Its head applies LayerNorm, a 512-wide Tanh layer and a linear layer that outputs a softmax over the 1000 ImageNet classes.

**The input size is fixed.** Window and grid partitions need 224×224 inputs, so every image is resized (shorter side to 235 px, bicubic) and centre-cropped to 224×224 (`crop_pct` 0.95) before normalisation with the ImageNet mean and standard deviation.

**The default path really adapts the model:** it downloads a pinned 400-image CIFAR-10 subset (`frog` and `truck`), measures two baselines on a held-out split — always answering the training majority, and mapping the ImageNet head's frog and truck classes to the two labels without any training — then replaces the head with a two-class layer, fine-tunes, scores the result on the same held-out split, classifies a further unseen split, exports the changed tensors as a SafeTensors adapter, and reloads that adapter onto a fresh copy of the verified base model to check that it reproduces the same predictions. Every number you see is measured in this notebook runtime.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream model revision; download and digest-verify a pinned labelled dataset and validate it before any model runs; read ImageNet top-5 predictions; build a zero-shot baseline by mapping ImageNet classes onto task labels; see what a closed-set classifier answers for blank and noise images; split a dataset with stratification; fine-tune with cross-entropy; compare accuracy and balanced accuracy against both baselines on a held-out split; classify unseen images; and export, reload and verify the adapter.

**This notebook does not demonstrate:** ImageNet benchmark results (nothing here re-scores the ImageNet validation set); CIFAR-10 benchmark results (the tutorial uses two classes and a few hundred images); real-world frog or vehicle recognition (CIFAR-10 images are 32×32 thumbnails, upsampled here to 224 px); open-set recognition or a reject option (every image receives one of the known classes); calibrated probabilities; object detection, segmentation or multi-label tagging.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). A CUDA GPU such as a Colab or Kaggle T4 is the documented runtime for the fine-tuning stages and is used automatically when present; the notebook also runs on CPU, more slowly. Runtimes are not measured in this revision. The pinned `torch==2.14.0` wheel is the largest download; the checkpoint is about 124 MB.
- **Knowledge:** basic Python and PIL; what a softmax over classes is; convolution and self-attention at the level of "local versus global context"; accuracy, balanced accuracy and a confusion matrix; why a baseline is needed before a score means anything.
- **Data:** the default path downloads one pinned archive, `CIFAR-10-subset.zip` from the Hugging Face dataset `Cleanlab/cifar-10-subset` (MIT licence, 986,707 bytes, verified by SHA-256 before it is opened), and keeps its `frog` and `truck` folders. BYOD is optional and off by default. Expected BYOD input: one image, or a directory (or `.zip`) of class folders, `<class>/<image>`, with at least 2 images per class. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so; uploaded inputs stay in this runtime and are not sent to any inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `timm/maxvit_tiny_tf_224.in1k` snapshot (~124 MB in total) at revision `041f2cce4d74…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `torchvision`, `timm`, `numpy`, `PIL` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'timm==1.0.29',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'maxvit-classification-pipeline',
    'repository_revision': '6b59382903196e56a299ae795d4d11d02c244be8',
    'embedded_module': 'src/maxvit_classification_pipeline/pipeline.py',
    'embedded_modules': ['src/maxvit_classification_pipeline/data.py', 'src/maxvit_classification_pipeline/pipeline.py'],
    'module_sha256': '7678e304d2dfa66bba3dcaab8df208b440753545bd7eea71372a80a47c164e73',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, torchvision, timm, numpy, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__, 'timm': timm.__version__, 'numpy': numpy.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/maxvit_classification_pipeline/` @ `6b5938290319`)

The next 2 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/2:** `src/maxvit_classification_pipeline/data.py`

In [ ]:
"""Labelled image-classification data: the pinned tutorial sample, class-folder readers, validation, splits.

Nothing here needs torch — Pillow and numpy only, plus ``huggingface_hub`` when the sample archive is
downloaded. Two sources produce the same record shape, ``{"id": str, "image": PIL.Image, "label": str}``:

* ``fetch_sample_archive`` + ``read_class_archive``: the pinned ``Cleanlab/cifar-10-subset`` archive
  (MIT licence), checked against a recorded byte size and SHA-256 before any member is opened;
* ``read_class_folder``: a caller's own directory of ``<class>/<image>`` files (BYOD).

Both refuse absolute member paths, ``..`` segments and oversized archives before decoding anything, and
``validate_dataset`` checks the result before any model runs.
"""

from __future__ import annotations

import hashlib
import io
import zipfile
from collections.abc import Callable, Mapping, Sequence
from pathlib import Path, PurePosixPath
from typing import Any

import numpy as np
from PIL import Image

# The tutorial sample: a CIFAR-10 subset published as one zip archive of class folders. The revision,
# size and digest are those of the archive at that Hub commit; `fetch_sample_archive` refuses any other bytes.
SAMPLE_DATASET_ID = "Cleanlab/cifar-10-subset"
SAMPLE_DATASET_REVISION = "bb5a7aabf1d14d2d1e3e49d0d8f917bda3622f75"
SAMPLE_DATASET_FILE = "CIFAR-10-subset.zip"
SAMPLE_DATASET_BYTES = 986707
SAMPLE_DATASET_SHA256 = "66f90a4f87d865e8eb653b62f10e754684075a32314177de76832349d4b1fb19"
SAMPLE_DATASET_LICENSE = "mit"
SAMPLE_CLASSES: tuple[str, ...] = ("frog", "truck")

IMAGE_SUFFIXES = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
MAX_ARCHIVE_MEMBERS = 20000
MAX_ARCHIVE_BYTES = 512 * 1024 * 1024  # total uncompressed size, checked from the zip directory
MAX_RECORDS = 5000
MAX_CLASSES = 1000
MIN_PER_CLASS = 2
MIN_IMAGE_SIDE = 8
MAX_IMAGE_SIDE = 4096


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_archive(
    path: str | Path, *, size: int = SAMPLE_DATASET_BYTES, sha256: str = SAMPLE_DATASET_SHA256
) -> dict[str, Any]:
    """Check an archive's byte size and SHA-256; raise naming the first mismatch."""
    archive = Path(path)
    if not archive.is_file():
        raise FileNotFoundError(f"archive not found: {archive}")
    actual_size = archive.stat().st_size
    if actual_size != size:
        raise ValueError(f"{archive.name}: size {actual_size} != recorded {size}")
    digest = _sha256_file(archive)
    if digest != sha256:
        raise ValueError(f"{archive.name}: sha256 {digest} != recorded {sha256}")
    return {"path": str(archive), "bytes": actual_size, "sha256": digest}


def _hub_dataset_download(root: Path) -> None:
    from huggingface_hub import hf_hub_download

    hf_hub_download(
        SAMPLE_DATASET_ID,
        SAMPLE_DATASET_FILE,
        repo_type="dataset",
        revision=SAMPLE_DATASET_REVISION,
        local_dir=str(root),
    )


def fetch_sample_archive(
    directory: str | Path,
    *,
    allow_download: bool = False,
    downloader: Callable[[Path], None] | None = None,
) -> dict[str, Any]:
    """Return the verified tutorial archive under ``directory``, downloading it at the pinned revision if
    it is absent and ``allow_download`` is set. The archive is verified whether or not it was downloaded."""
    root = Path(directory)
    root.mkdir(parents=True, exist_ok=True)
    path = root / SAMPLE_DATASET_FILE
    fetched = False
    if not path.is_file():
        if not allow_download:
            raise FileNotFoundError(
                f"{path} is absent; pass allow_download=True to fetch {SAMPLE_DATASET_ID}"
            )
        (downloader or _hub_dataset_download)(root)
        fetched = True
    info = verify_archive(path, size=SAMPLE_DATASET_BYTES, sha256=SAMPLE_DATASET_SHA256)
    return {
        **info,
        "fetched": fetched,
        "dataset_id": SAMPLE_DATASET_ID,
        "revision": SAMPLE_DATASET_REVISION,
        "license": SAMPLE_DATASET_LICENSE,
    }


def _safe_member(name: str) -> PurePosixPath:
    member = PurePosixPath(name)
    if (
        name.startswith(("/", "\\"))
        or member.is_absolute()
        or ".." in member.parts
        or ":" in (member.parts[0] if member.parts else "")
    ):
        raise ValueError(f"archive member {name!r} is absolute or leaves the archive root")
    return member


def _open_image(data: bytes, where: str) -> Image.Image:
    try:
        with Image.open(io.BytesIO(data)) as handle:
            width, height = handle.size
            if max(width, height) > MAX_IMAGE_SIDE:
                raise ValueError(
                    f"{where}: image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}"
                )
            return handle.convert("RGB")
    except (OSError, SyntaxError) as exc:
        raise ValueError(f"{where}: not a decodable image ({exc})") from exc


def _select(
    by_class: Mapping[str, list[Any]], classes: Sequence[str] | None, max_per_class: int | None, seed: int
) -> dict[str, list[Any]]:
    if classes is not None:
        missing = [name for name in classes if name not in by_class]
        if missing:
            raise ValueError(f"classes {missing} have no image; found {sorted(by_class)}")
        by_class = {name: by_class[name] for name in classes}
    if max_per_class is not None:
        if isinstance(max_per_class, bool) or not isinstance(max_per_class, int) or max_per_class < 1:
            raise ValueError(f"max_per_class must be a positive int, got {max_per_class!r}")
        rng = np.random.default_rng(seed)
        by_class = {
            name: [items[int(i)] for i in sorted(rng.permutation(len(items))[:max_per_class])]
            for name, items in sorted(by_class.items())
        }
    return dict(by_class)


def read_class_archive(
    path: str | Path,
    *,
    classes: Sequence[str] | None = None,
    max_per_class: int | None = None,
    seed: int = 0,
) -> list[dict[str, Any]]:
    """Read ``{"id", "image", "label"}`` records from a zip of class folders (``.../<class>/<image>``).

    The label is the image's parent folder name. Member names are checked (no absolute paths, no ``..``)
    and the member count and total uncompressed size are bounded before any member is decompressed.
    ``classes`` keeps only those folders and fails if one is absent; ``max_per_class`` keeps a seeded
    sample of that many images per class.
    """
    with zipfile.ZipFile(path) as archive:
        infos = archive.infolist()
        if len(infos) > MAX_ARCHIVE_MEMBERS:
            raise ValueError(f"archive has {len(infos)} members > MAX_ARCHIVE_MEMBERS {MAX_ARCHIVE_MEMBERS}")
        total = sum(info.file_size for info in infos)
        if total > MAX_ARCHIVE_BYTES:
            raise ValueError(f"archive expands to {total} bytes > MAX_ARCHIVE_BYTES {MAX_ARCHIVE_BYTES}")
        by_class: dict[str, list[str]] = {}
        for info in infos:
            member = _safe_member(info.filename)
            if info.is_dir() or "__MACOSX" in member.parts or member.name.startswith("."):
                continue
            if member.suffix.lower() not in IMAGE_SUFFIXES or len(member.parts) < 2:
                continue
            by_class.setdefault(member.parts[-2], []).append(info.filename)
        if not by_class:
            raise ValueError("archive holds no image inside a class folder")
        selected = _select({k: sorted(v) for k, v in by_class.items()}, classes, max_per_class, seed)
        return [
            {"id": name, "image": _open_image(archive.read(name), name), "label": label}
            for label, names in selected.items()
            for name in names
        ]


def read_class_folder(
    directory: str | Path,
    *,
    classes: Sequence[str] | None = None,
    max_per_class: int | None = None,
    seed: int = 0,
) -> list[dict[str, Any]]:
    """Read records from ``<directory>/<class>/<image>``; links that leave ``directory`` are refused."""
    root = Path(directory).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"{root} is not a directory")
    by_class: dict[str, list[Path]] = {}
    for class_dir in sorted(p for p in root.iterdir() if p.is_dir() and not p.name.startswith(".")):
        for file in sorted(class_dir.iterdir()):
            if file.suffix.lower() not in IMAGE_SUFFIXES or file.name.startswith("."):
                continue
            resolved = file.resolve()
            if root not in resolved.parents:
                raise ValueError(f"{file} resolves outside {root}")
            by_class.setdefault(class_dir.name, []).append(resolved)
    if not by_class:
        raise ValueError(f"{root} holds no image inside a class folder")
    selected = _select(by_class, classes, max_per_class, seed)
    return [
        {
            "id": str(file.relative_to(root)),
            "image": _open_image(file.read_bytes(), str(file.relative_to(root))),
            "label": label,
        }
        for label, files in selected.items()
        for file in files
    ]


def _check_class_names(class_names: Sequence[str]) -> tuple[str, ...]:
    names = tuple(class_names)
    if not 2 <= len(names) <= MAX_CLASSES:
        raise ValueError(f"class_names must hold 2..{MAX_CLASSES} names, got {len(names)}")
    for name in names:
        if not isinstance(name, str) or not name.strip():
            raise ValueError(f"class names must be non-empty strings, got {name!r}")
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        raise ValueError(f"class_names contains duplicates: {duplicates}")
    return names


def _pixel_digest(image: Image.Image) -> str:
    return _sha256_bytes(image.convert("RGB").tobytes() + repr(image.size).encode())


def validate_dataset(
    records: Sequence[Mapping[str, Any]], class_names: Sequence[str], *, epochs: int = 5
) -> dict[str, Any]:
    """Validation stage: raise on the first broken record, else return the dataset manifest.

    A class with fewer than ``MIN_PER_CLASS`` images is an error, because it cannot be split. Findings,
    which do not stop the run, report class imbalance above 2:1 and exact pixel duplicates; a duplicate
    inflates the held-out score when one copy lands on each side of a split.
    """
    names = _check_class_names(class_names)
    if not records:
        raise ValueError("dataset must hold at least one record")
    if len(records) > MAX_RECORDS:
        raise ValueError(f"dataset holds {len(records)} records > MAX_RECORDS {MAX_RECORDS}")
    if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 100:
        raise ValueError(f"epochs must be an int in 1..100, got {epochs!r}")
    counts = dict.fromkeys(names, 0)
    sides: list[int] = []
    digests: dict[str, list[str]] = {}
    for idx, record in enumerate(records):
        if not isinstance(record, Mapping) or not {"image", "label"} <= set(record):
            raise ValueError(f"record {idx} must be a mapping with 'image' and 'label'")
        image = record["image"]
        if not isinstance(image, Image.Image):
            raise TypeError(f"record {idx}: image must be a PIL.Image.Image, got {type(image).__name__}")
        if min(image.size) < MIN_IMAGE_SIDE or max(image.size) > MAX_IMAGE_SIDE:
            raise ValueError(
                f"record {idx}: image size {image.size} outside {MIN_IMAGE_SIDE}..{MAX_IMAGE_SIDE} px"
            )
        if record["label"] not in counts:
            raise ValueError(
                f"record {idx} has unknown class {record['label']!r}; expected one of {list(names)}"
            )
        counts[record["label"]] += 1
        sides.extend(image.size)
        digests.setdefault(_pixel_digest(image), []).append(str(record.get("id", idx)))
    short = {name: n for name, n in counts.items() if n < MIN_PER_CLASS}
    if short:
        raise ValueError(
            f"every class needs at least {MIN_PER_CLASS} images to split; short classes: {short}"
        )
    findings = []
    if max(counts.values()) > 2 * min(counts.values()):
        findings.append(f"class imbalance above 2:1: {counts}")
    duplicates = [ids for ids in digests.values() if len(ids) > 1]
    if duplicates:
        findings.append(f"{len(duplicates)} group(s) of pixel-identical images, e.g. {duplicates[0][:3]}")
    return {
        "n_records": len(records),
        "class_names": list(names),
        "images_per_class": counts,
        "image_side_px": [min(sides), max(sides)],
        "duplicate_groups": len(duplicates),
        "epochs": epochs,
        "findings": findings,
        "verdict": "accepted",
    }


def split_dataset(
    records: Sequence[Mapping[str, Any]], *, train_fraction: float = 0.7, seed: int = 0
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Stratified, seeded split: each class is shuffled and cut at ``train_fraction``, keeping at least one
    image of every class on each side. A random split assumes independent records; images that share a
    source photograph, scene or session must be split by that group instead, or the held-out score leaks."""
    if not 0.0 < train_fraction < 1.0:
        raise ValueError(f"train_fraction must be between 0 and 1, got {train_fraction}")
    by_class: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        by_class.setdefault(record["label"], []).append(dict(record))
    rng = np.random.default_rng(seed)
    train: list[dict[str, Any]] = []
    held_out: list[dict[str, Any]] = []
    for label in sorted(by_class):
        items = by_class[label]
        if len(items) < 2:
            raise ValueError(f"class {label!r} has {len(items)} record(s); at least 2 are needed to split")
        order = rng.permutation(len(items))
        n_train = min(len(items) - 1, max(1, round(len(items) * train_fraction)))
        train.extend(items[int(i)] for i in order[:n_train])
        held_out.extend(items[int(i)] for i in order[n_train:])
    return train, held_out


def blank_image(width: int = 224, height: int = 224) -> Image.Image:
    """A featureless white image: a closed-set classifier still assigns it one of its classes."""
    return Image.new("RGB", (width, height), (255, 255, 255))


def noise_image(seed: int = 0, width: int = 224, height: int = 224) -> Image.Image:
    """Uniform RGB noise: structure-free input, for the same reason as ``blank_image``."""
    rng = np.random.default_rng(seed)
    return Image.fromarray(rng.integers(0, 256, (height, width, 3), dtype=np.uint8))

**Module 2/2:** `src/maxvit_classification_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""ImageNet-1k classification and bounded fine-tuning with the ``timm/maxvit_tiny_tf_224.in1k`` checkpoint.

MaxViT-Tiny is a hybrid network: every block is an MBConv convolution (with BatchNorm) followed by window
and grid self-attention (with LayerNorm); its input size is fixed at 224x224.

The class loads weights only from a digest-verified local snapshot (``weights/<key>/``). The architecture
comes from the pinned ``timm`` release, built with ``pretrained=False`` so that timm fetches nothing, and
the SafeTensors state dict is loaded with ``strict=True``. Preprocessing is the checkpoint's own
``pretrained_cfg`` (resize, centre crop, ImageNet normalisation) resolved through ``timm.data``.

Until ``tools/pin_snapshot.py`` has recorded an immutable revision and every file's SHA-256, the package
refuses to stage, verify or load weights: an unpinned snapshot is never trusted.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

# standalone rewrite (build_notebook.py): `from .data import validate_dataset` removed — names are kernel globals defined by the carried modules

MODEL_ID = "timm/maxvit_tiny_tf_224.in1k"
MODEL_REVISION = "041f2cce4d74c7539d63aa9fb85786e78072d487"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "maxvit-tiny-tf-224-in1k"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
ARTIFACT_FORMAT = "maxvit-adapter-v1"
UNPINNED = "unpinned"
PIN_COMMAND = "python tools/pin_snapshot.py"
WEIGHTS_FILE = "model.safetensors"
CONFIG_FILE = "config.json"

NUM_IMAGENET_CLASSES = 1000
MIN_IMAGE_SIDE = 8
MAX_IMAGE_SIDE = 4096
MAX_BATCH = 64  # images per predict() call and per forward pass
MAX_CLASSES = 1000
DEFAULT_TOP_K = 5
DECISION_RULE = (
    "argmax"  # the reported label is the softmax argmax; there is no threshold and no reject option
)

# ImageNet-1k classes that fall inside each tutorial class, for the zero-shot baseline. CIFAR-10 defines
# "truck" as big trucks only and excludes pickup trucks, so pickup (717), minivan (656) and police van
# (734) are left out; "frog" takes all three ImageNet frog classes.
IMAGENET_GROUPS: dict[str, tuple[int, ...]] = {
    "frog": (30, 31, 32),  # bullfrog, tree frog, tailed frog
    "truck": (555, 569, 675, 864, 867),  # fire engine, garbage truck, moving van, tow truck, trailer truck
}

# Bounded tutorial fine-tuning defaults: AdamW on cross-entropy, float32, no augmentation.
DEFAULT_EPOCHS = 5
DEFAULT_BATCH_SIZE = 16
DEFAULT_LEARNING_RATE = 1e-4
DEFAULT_WEIGHT_DECAY = 0.01
DEFAULT_SEED = 20260925


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def is_pinned() -> bool:
    """True once MODEL_REVISION names an immutable 40-hex commit."""
    revision = MODEL_REVISION
    return len(revision) == 40 and all(c in "0123456789abcdef" for c in revision)


def _require_pinned(action: str) -> None:
    if not is_pinned():
        raise RuntimeError(
            f"refusing to {action}: {MODEL_ID} has no pinned revision yet (MODEL_REVISION = "
            f"{MODEL_REVISION!r}); run `{PIN_COMMAND}` to record the commit and every file's SHA-256"
        )


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    _require_pinned("verify the snapshot")
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        if not entry.get("sha256"):
            raise ValueError(f"{entry['path']}: manifest records no sha256; run `{PIN_COMMAND}`")
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
        "weights_sha256": next((e["sha256"] for e in manifest["files"] if e["path"] == WEIGHTS_FILE), None),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    _require_pinned("stage weights")
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at "
            f"{MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


# --------------------------------------------------------------------------- metrics


def classification_metrics(
    predicted: Sequence[str],
    truth: Sequence[str],
    class_names: Sequence[str],
    *,
    majority_class: str | None = None,
) -> dict[str, Any]:
    """Accuracy, balanced accuracy, per-class recall and the confusion matrix, plus the accuracy of always
    predicting ``majority_class`` (pass the training majority, so the baseline is not tuned on the
    evaluated set)."""
    if len(predicted) != len(truth):
        raise ValueError(f"{len(predicted)} predictions but {len(truth)} labels")
    if not truth:
        raise ValueError("at least one labelled item is needed")
    names = list(class_names)
    index = {name: i for i, name in enumerate(names)}
    matrix = np.zeros((len(names), len(names)), dtype=int)
    for p, t in zip(predicted, truth, strict=True):
        matrix[index[t], index[p]] += 1
    support = matrix.sum(axis=1)
    recall = {
        name: (float(matrix[i, i] / support[i]) if support[i] else None) for i, name in enumerate(names)
    }
    present = [value for value in recall.values() if value is not None]
    result = {
        "accuracy": float(np.trace(matrix) / matrix.sum()),
        "balanced_accuracy": float(np.mean(present)),
        "per_class_recall": recall,
        "support": {name: int(n) for name, n in zip(names, support, strict=True)},
        "confusion_matrix": {"labels": names, "rows_true_cols_predicted": matrix.tolist()},
        "n": int(matrix.sum()),
    }
    if majority_class is not None:
        if majority_class not in index:
            raise ValueError(f"majority_class {majority_class!r} is not one of {names}")
        result["majority_class"] = majority_class
        result["majority_baseline_accuracy"] = float(
            sum(1 for t in truth if t == majority_class) / len(truth)
        )
    return result


def majority_class(records: Sequence[Mapping[str, Any]]) -> str:
    """The most frequent label, ties broken by name, for the majority-class baseline."""
    counts: dict[str, int] = {}
    for record in records:
        counts[record["label"]] = counts.get(record["label"], 0) + 1
    return sorted(counts.items(), key=lambda item: (-item[1], item[0]))[0][0]


# --------------------------------------------------------------------------- input validation


def _check_images(images: Any) -> list[Image.Image]:
    if isinstance(images, Image.Image):
        images = [images]
    if not isinstance(images, Sequence) or isinstance(images, str | bytes):
        raise TypeError("images must be a PIL.Image.Image or a sequence of them")
    if not 1 <= len(images) <= MAX_BATCH:
        raise ValueError(f"batch size must be between 1 and MAX_BATCH={MAX_BATCH}, got {len(images)}")
    for image in images:
        if not isinstance(image, Image.Image):
            raise TypeError(f"each image must be a PIL.Image.Image, got {type(image).__name__}")
        if min(image.size) < MIN_IMAGE_SIDE or max(image.size) > MAX_IMAGE_SIDE:
            raise ValueError(
                f"image size {image.size} outside MIN_IMAGE_SIDE..MAX_IMAGE_SIDE = "
                f"{MIN_IMAGE_SIDE}..{MAX_IMAGE_SIDE} px"
            )
    return [image.convert("RGB") for image in images]


def _check_top_k(top_k: Any, n_classes: int) -> int:
    if isinstance(top_k, bool) or not isinstance(top_k, int):
        raise TypeError("top_k must be an int")
    if not 1 <= top_k <= n_classes:
        raise ValueError(f"top_k must be between 1 and {n_classes}, got {top_k}")
    return top_k


def _check_class_names(class_names: Sequence[str]) -> tuple[str, ...]:
    names = tuple(class_names)
    if not 2 <= len(names) <= MAX_CLASSES:
        raise ValueError(f"class_names must hold 2..{MAX_CLASSES} names, got {len(names)}")
    for name in names:
        if not isinstance(name, str) or not name.strip():
            raise ValueError(f"class names must be non-empty strings, got {name!r}")
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        raise ValueError(f"class_names contains duplicates: {duplicates}")
    return names


INPUT_SCHEMA: dict[str, Any] = {
    "input": "PIL.Image.Image or a sequence of them; any mode, converted to RGB",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "batch": [1, MAX_BATCH],
    "top_k": [1, NUM_IMAGENET_CLASSES],
    "decision_rule": DECISION_RULE,
    "preprocessing": (
        "RGB; bicubic resize of the shorter side to 235 px, centre crop 224x224 (crop_pct 0.95), ImageNet "
        "mean/std normalisation, from the checkpoint's pretrained_cfg; the input size is fixed at 224x224"
    ),
}


def validate_inputs(
    images: Image.Image | Sequence[Image.Image],
    top_k: int = DEFAULT_TOP_K,
    *,
    names: Sequence[str] | None = None,
    n_classes: int = NUM_IMAGENET_CLASSES,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, request, verdict)."""
    checked = _check_images(images)
    _check_top_k(top_k, n_classes)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per image")
    originals = [images] if isinstance(images, Image.Image) else list(images)
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[i] if names else f"image-{i}", "mode": image.mode, "size": list(image.size)}
            for i, image in enumerate(originals)
        ],
        "top_k": top_k,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    truth: Sequence[str] | None = None,
    *,
    groups: Mapping[str, Sequence[int]] | None = None,
    sample_kind: str = "sample",
) -> dict[str, Any]:
    """Single-batch evaluation stage for ImageNet-head predictions.

    With ``truth`` (one tutorial class name per image) and ``groups`` (the ImageNet indices each class
    covers), a prediction counts as correct when its top-1 or any of its top-k indices falls inside the
    image's group. Without them the verdict is ``not-measurable`` and the report names the missing data.
    """
    predictions = list(result["predictions"])
    base = {
        "task": f"{NUM_IMAGENET_CLASSES}-class ImageNet-1k single-label classification",
        "decision_rule": result.get("decision_rule", DECISION_RULE),
        "sample_kind": sample_kind,
        "n_predictions": len(predictions),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if truth is None or groups is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth class was supplied for the evaluated images",
            "needs": (
                "labelled images from your own domain, scored with accuracy and balanced accuracy against "
                "the "
                "majority-class baseline of the training split"
            ),
        }
    if len(truth) != len(predictions):
        raise ValueError(f"{len(predictions)} predictions but {len(truth)} labels")
    top_k = int(result.get("top_k", DEFAULT_TOP_K))
    metrics = []
    for k in sorted({1, top_k}):
        hits = [
            any(item["index"] in groups[label] for item in pred["top_k"][:k])
            for pred, label in zip(predictions, truth, strict=True)
        ]
        metrics.append(
            {
                "id": "group_hit_rate",
                "k": k,
                "value": float(np.mean(hits)),
                "estimation": "one pass, no dispersion estimate",
            }
        )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": f"{len(predictions)} labelled image(s) scored against hand-chosen ImageNet class groups; "
        "not a benchmark",
        "needs": "a labelled evaluation set from the deployment domain for any generalisable accuracy claim",
    }


# --------------------------------------------------------------------------- backend (timm)
# Everything model-specific lives in this section: how the architecture is built, how the pinned state
# dict is loaded, which transform it expects, and which module is the classification head.

ARCHITECTURE = "maxvit_tiny_tf_224"
PRETRAINED_TAG = "in1k"
# The trainable head in a frozen fine-tune: timm's NormMlpClassifierHead (LayerNorm, a 512-to-512 Tanh
# pre-logits layer and the final ``fc``). Re-heading replaces ``head.fc`` only.
HEAD_MODULE = "head"


def build_model(num_classes: int = NUM_IMAGENET_CLASSES, **overrides: Any) -> Any:
    """timm's maxvit_tiny_tf_224 with random weights; ``pretrained=False`` so nothing is downloaded.
    ``overrides`` (for tests) shrink the architecture through timm's MaxxVit config overlay."""
    import timm

    return timm.create_model(
        f"{ARCHITECTURE}.{PRETRAINED_TAG}", pretrained=False, num_classes=num_classes, **overrides
    )


def eval_transform(model: Any) -> Callable[[Image.Image], Any]:
    """The checkpoint's evaluation preprocessing, resolved from the model's pretrained_cfg."""
    from timm.data import create_transform, resolve_model_data_config

    return create_transform(**resolve_model_data_config(model), is_training=False)


def imagenet_labels() -> tuple[str, ...]:
    """Human-readable ImageNet-1k class descriptions in index order, bundled with timm (no download)."""
    from timm.data import ImageNetInfo

    info = ImageNetInfo(subset="imagenet-1k")
    return tuple(info.index_to_description(i) for i in range(info.num_classes()))


def _load_pretrained(root: Path) -> Any:
    from safetensors.torch import load_file

    with open(root / CONFIG_FILE, encoding="utf-8") as fh:
        config = json.load(fh)
    named = f"{config['architecture']}.{config['pretrained_cfg']['tag']}"
    if named != f"{ARCHITECTURE}.{PRETRAINED_TAG}":
        raise ValueError(f"snapshot config names {named!r}, package pins {ARCHITECTURE}.{PRETRAINED_TAG}")
    model = build_model(NUM_IMAGENET_CLASSES)
    model.load_state_dict(load_file(str(root / WEIGHTS_FILE), device="cpu"), strict=True)
    return model


def _rehead(model: Any, num_classes: int) -> tuple[str, ...]:
    model.reset_classifier(num_classes)
    return (f"{HEAD_MODULE}.fc.",)


def _backbone_prefixes(model: Any) -> tuple[str, ...]:
    return tuple(f"{name}." for name, _ in model.named_children() if name != HEAD_MODULE)


# --------------------------------------------------------------------------- pipeline


def _hold_batchnorm(modules: Sequence[Any]) -> None:
    """Keep BatchNorm layers in eval mode, so frozen layers keep their running statistics."""
    import torch

    for module in modules:
        for sub in module.modules():
            if isinstance(sub, torch.nn.modules.batchnorm._BatchNorm):
                sub.eval()


@dataclass
class MaxViTPipeline:
    """ImageNet-1k classification, and transfer fine-tuning onto a caller's classes, over MaxViT-Tiny."""

    model: Any
    transform: Callable[[Image.Image], Any]
    device: str
    class_names: tuple[str, ...]
    source: str = "snapshot"
    base_state_digest: str | None = None
    adapted: bool = False
    reinitialised: tuple[str, ...] = field(default_factory=tuple)
    frozen_prefixes: tuple[str, ...] = field(default_factory=tuple)

    @property
    def imagenet_head(self) -> bool:
        return len(self.class_names) == NUM_IMAGENET_CLASSES and not self.reinitialised and not self.adapted

    @classmethod
    def from_components(
        cls,
        model: Any,
        *,
        class_names: Sequence[str],
        device: str = "cpu",
        transform: Callable[[Image.Image], Any] | None = None,
        source: str = "components",
    ) -> MaxViTPipeline:
        """Wrap an already-built model; the transform defaults to the model's own evaluation transform."""
        return cls(
            model=model.to(device).eval(),
            transform=transform or eval_transform(model),
            device=device,
            class_names=tuple(class_names),
            source=source,
        )

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        class_names: Sequence[str] | None = None,
        seed: int = DEFAULT_SEED,
    ) -> MaxViTPipeline:
        """Stage (only with ``allow_download``), verify and load the pinned checkpoint. With ``class_names``,
        replace the 1000-class head with a new one for those classes, initialised under ``seed``."""
        _require_pinned("load the model")
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(
                f"no snapshot manifest at {root}; stage {MODEL_ID} under weights/{MODEL_KEY} "
                "(allow_download=True fetches the manifest-listed files)"
            )
        stage_missing_files(root, allow_download=allow_download)
        snapshot = verify_snapshot(root)
        names = _check_class_names(class_names) if class_names is not None else None

        import torch

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        model = _load_pretrained(root)
        reinitialised: tuple[str, ...] = ()
        if names is not None:
            torch.manual_seed(seed)
            reinitialised = _rehead(model, len(names))
        pipe = cls.from_components(
            model,
            class_names=names or imagenet_labels(),
            device=resolved_device,
            transform=eval_transform(model),
            source=str(root),
        )
        pipe.base_state_digest = snapshot["weights_sha256"]
        pipe.reinitialised = reinitialised
        return pipe

    def _probabilities(self, images: Sequence[Image.Image]) -> Any:
        import torch

        was_training = self.model.training
        self.model.eval()
        outputs = []
        with torch.inference_mode():
            for start in range(0, len(images), MAX_BATCH):
                batch = torch.stack(
                    [self.transform(image) for image in images[start : start + MAX_BATCH]]
                ).to(self.device)
                outputs.append(torch.softmax(self.model(batch).float(), dim=-1).cpu())
        if was_training:
            self.model.train()
        probabilities = torch.cat(outputs)
        if probabilities.shape != (len(images), len(self.class_names)):
            raise RuntimeError(
                f"model returned {tuple(probabilities.shape)}, expected ({len(images)}, "
                f"{len(self.class_names)})"
            )
        return probabilities

    def predict(
        self, images: Image.Image | Sequence[Image.Image], top_k: int | None = None
    ) -> dict[str, Any]:
        """Classify images; ``score`` is a softmax over this pipeline's classes, not a calibrated value."""
        import torch

        batch = _check_images(images)
        resolved = (
            min(DEFAULT_TOP_K, len(self.class_names))
            if top_k is None
            else _check_top_k(top_k, len(self.class_names))
        )
        values, indices = torch.topk(self._probabilities(batch), k=resolved, dim=-1)
        predictions = []
        for row_values, row_indices in zip(values.tolist(), indices.tolist(), strict=True):
            ranked = [
                {"label": self.class_names[i], "index": int(i), "score": float(s)}
                for s, i in zip(row_values, row_indices, strict=True)
            ]
            predictions.append(
                {
                    "predicted_label": ranked[0]["label"],
                    "predicted_index": ranked[0]["index"],
                    "top_k": ranked,
                }
            )
        return {
            "predictions": predictions,
            "top_k": resolved,
            "decision_rule": DECISION_RULE,
            "class_names_count": len(self.class_names),
            "adapted": self.adapted,
            "device": self.device,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def zero_shot_evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        groups: Mapping[str, Sequence[int]] = IMAGENET_GROUPS,
        *,
        majority: str | None = None,
    ) -> dict[str, Any]:
        """Score the ImageNet head on tutorial classes without training: each image goes to the class whose
        ImageNet group holds the most softmax mass. Needs the unmodified 1000-class head."""
        if not self.imagenet_head:
            raise RuntimeError("zero_shot_evaluate needs the pretrained 1000-class ImageNet head")
        names = list(groups)
        missing = sorted({r["label"] for r in records} - set(names))
        if missing:
            raise ValueError(f"records carry labels with no ImageNet group: {missing}")
        probabilities = self._probabilities([r["image"].convert("RGB") for r in records])
        mass = np.stack([probabilities[:, list(groups[name])].sum(dim=1).numpy() for name in names], axis=1)
        predicted = [names[int(i)] for i in mass.argmax(axis=1)]
        truth = [r["label"] for r in records]
        metrics = classification_metrics(predicted, truth, names, majority_class=majority)
        return {
            **metrics,
            "rule": "argmax of summed ImageNet softmax mass per class group",
            "groups": {name: list(ids) for name, ids in groups.items()},
            "mean_group_mass": float(mass.sum(axis=1).mean()),
            "adapted": False,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def finetune(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        epochs: int = DEFAULT_EPOCHS,
        batch_size: int = DEFAULT_BATCH_SIZE,
        learning_rate: float = DEFAULT_LEARNING_RATE,
        weight_decay: float = DEFAULT_WEIGHT_DECAY,
        seed: int = DEFAULT_SEED,
        freeze_backbone: bool = False,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded cross-entropy fine-tuning on ``records`` (``{"image", "label"}``), mutating this pipeline.

        With ``freeze_backbone`` only the head trains and every BatchNorm layer outside it is held in eval
        mode, so the backbone keeps its weights and running statistics; otherwise the whole network trains.
        The inputs use the evaluation transform: there is no data augmentation.
        """
        import torch
        import torch.nn.functional as F

        validate_dataset(records, self.class_names, epochs=epochs)
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or batch_size < 1:
            raise ValueError(f"batch_size must be a positive int, got {batch_size!r}")
        if (
            isinstance(learning_rate, bool)
            or not isinstance(learning_rate, int | float)
            or not 0.0 < float(learning_rate) <= 1.0
        ):
            raise ValueError(f"learning_rate must be a number in (0, 1], got {learning_rate!r}")

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        rng = np.random.default_rng(seed)
        backbone = _backbone_prefixes(self.model)
        for name, parameter in self.model.named_parameters():
            parameter.requires_grad = not (freeze_backbone and name.startswith(backbone))
        self.frozen_prefixes = backbone if freeze_backbone else ()
        trainable = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable, lr=float(learning_rate), weight_decay=float(weight_decay))
        class_to_id = {name: i for i, name in enumerate(self.class_names)}
        frozen_modules = [
            module for name, module in self.model.named_children() if f"{name}." in self.frozen_prefixes
        ]

        epoch_losses: list[float] = []
        for epoch in range(epochs):
            self.model.train()
            _hold_batchnorm(frozen_modules)
            order = rng.permutation(len(records))
            running, seen = 0.0, 0
            for start in range(0, len(records), batch_size):
                batch = [records[int(i)] for i in order[start : start + batch_size]]
                inputs = torch.stack([self.transform(r["image"].convert("RGB")) for r in batch]).to(
                    self.device
                )
                targets = torch.tensor(
                    [class_to_id[r["label"]] for r in batch], dtype=torch.long, device=self.device
                )
                optimizer.zero_grad(set_to_none=True)
                loss = F.cross_entropy(self.model(inputs), targets)
                loss.backward()
                optimizer.step()
                running += float(loss.detach().cpu()) * len(batch)
                seen += len(batch)
            epoch_losses.append(running / max(1, seen))
            if progress is not None:
                progress({"epoch": epoch + 1, "epochs": epochs, "loss": epoch_losses[-1]})
        self.model.eval()
        self.adapted = True
        return {
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": float(learning_rate),
            "weight_decay": float(weight_decay),
            "optimizer": "AdamW",
            "loss": "cross-entropy",
            "augmentation": "none (evaluation transform)",
            "seed": seed,
            "precision": "float32",
            "freeze_backbone": freeze_backbone,
            "frozen_prefixes": list(self.frozen_prefixes),
            "trainable_parameters": sum(p.numel() for p in trainable),
            "total_parameters": sum(p.numel() for p in self.model.parameters()),
            "epoch_losses": epoch_losses,
            "final_loss": epoch_losses[-1] if epoch_losses else None,
            "device": self.device,
            "class_names": list(self.class_names),
            "reinitialised_prefixes": list(self.reinitialised),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def evaluate(
        self, records: Sequence[Mapping[str, Any]], *, majority: str | None = None
    ) -> dict[str, Any]:
        """Score labelled records with this pipeline's head: accuracy, balanced accuracy, per-class recall,
        confusion matrix, and the accuracy of always answering ``majority`` (pass the training majority)."""
        unknown = sorted({r["label"] for r in records} - set(self.class_names))
        if unknown:
            raise ValueError(f"records carry labels outside this pipeline's classes: {unknown}")
        images = [r["image"].convert("RGB") for r in records]
        probabilities = self._probabilities(images)
        predicted = [self.class_names[int(i)] for i in probabilities.argmax(dim=-1).tolist()]
        metrics = classification_metrics(
            predicted, [r["label"] for r in records], self.class_names, majority_class=majority
        )
        return {
            **metrics,
            "mean_top1_score": float(probabilities.max(dim=-1).values.mean()),
            "adapted": self.adapted,
            "estimation": f"one pass over {len(records)} held-out images; no resampling, no dispersion "
            "estimate",
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def save_artifact(self, path: str | Path, *, notes: str | None = None) -> dict[str, Any]:
        """Write the adapted tensors as one SafeTensors file with the provenance in its metadata.

        Tensors under ``frozen_prefixes`` are left out: they equal the verified base checkpoint, which
        ``load_artifact`` loads first. The artifact is therefore an adapter bound to the base checkpoint.
        """
        from safetensors.torch import save_file

        if not self.adapted:
            raise RuntimeError("nothing to export: the pipeline has not been fine-tuned")
        artifact_path = Path(path)
        artifact_path.parent.mkdir(parents=True, exist_ok=True)
        tensors = {
            name: value.detach().cpu().contiguous().clone()
            for name, value in self.model.state_dict().items()
            if not any(name.startswith(prefix) for prefix in self.frozen_prefixes)
        }
        metadata = {
            "format": ARTIFACT_FORMAT,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "model_key": MODEL_KEY,
            "class_names": json.dumps(list(self.class_names)),
            "frozen_prefixes": json.dumps(list(self.frozen_prefixes)),
            "base_state_digest": self.base_state_digest or "",
            "notes": notes or "",
        }
        save_file(tensors, str(artifact_path), metadata=metadata)
        return {
            "path": str(artifact_path),
            "bytes": artifact_path.stat().st_size,
            "sha256": _sha256(artifact_path),
            "format": ARTIFACT_FORMAT,
            "class_names": list(self.class_names),
            "tensors": len(tensors),
            "frozen_prefixes": list(self.frozen_prefixes),
            "base_state_digest": self.base_state_digest,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    @staticmethod
    def read_artifact_metadata(path: str | Path) -> dict[str, Any]:
        """Read and check the artifact's provenance header without loading any tensor."""
        from safetensors import safe_open

        with safe_open(str(path), framework="pt") as handle:
            metadata = dict(handle.metadata() or {})
        if metadata.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {metadata.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if (metadata.get("model_id"), metadata.get("model_revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError(
                f"artifact was built on {metadata.get('model_id')}@{metadata.get('model_revision')}, "
                f"package pins {MODEL_ID}@{MODEL_REVISION}"
            )
        if metadata.get("model_key") != MODEL_KEY:
            raise ValueError(
                f"artifact was built on {metadata.get('model_key')!r}, package pins {MODEL_KEY!r}"
            )
        return {
            **metadata,
            "class_names": json.loads(metadata["class_names"]),
            "frozen_prefixes": json.loads(metadata["frozen_prefixes"]),
        }

    def apply_artifact(self, path: str | Path) -> None:
        """Load adapter tensors onto this base, re-headed pipeline; refuse a tensor set that does not fit."""
        from safetensors.torch import load_file

        metadata = self.read_artifact_metadata(path)
        if tuple(metadata["class_names"]) != tuple(self.class_names):
            raise ValueError("artifact class_names differ from this pipeline's class_names")
        expected_base = metadata.get("base_state_digest") or None
        if expected_base and self.base_state_digest and expected_base != self.base_state_digest:
            raise ValueError("artifact was exported against a different base checkpoint digest")
        tensors = load_file(str(path), device="cpu")
        prefixes = tuple(metadata["frozen_prefixes"])
        result = self.model.load_state_dict(tensors, strict=False)
        if result.unexpected_keys:
            raise ValueError(
                f"artifact carries tensors the model does not have: {result.unexpected_keys[:5]}"
            )
        stray = [key for key in result.missing_keys if not any(key.startswith(p) for p in prefixes)]
        if stray:
            raise ValueError(f"artifact is missing trainable tensors: {stray[:5]}")
        self.model.eval()
        self.adapted = True
        self.frozen_prefixes = prefixes
        self.source = f"artifact:{Path(path).name}"

    @classmethod
    def load_artifact(
        cls, path: str | Path, *, weights_dir: str | Path | None = None, device: str | None = None
    ) -> MaxViTPipeline:
        """Rebuild an adapted pipeline: verified base checkpoint first, then the adapter tensors."""
        metadata = cls.read_artifact_metadata(path)
        pipe = cls.from_pretrained(
            device=device, weights_dir=weights_dir, class_names=metadata["class_names"]
        )
        pipe.apply_artifact(path)
        return pipe

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `041f2cce4d74…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `MaxViTPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "maxvit-tiny-tf-224-in1k",
  "modelId": "timm/maxvit_tiny_tf_224.in1k",
  "revision": "041f2cce4d74c7539d63aa9fb85786e78072d487",
  "files": [
    {
      "path": "README.md",
      "bytes": 22118,
      "sha256": "710144bf52c7f7fbe89276b75531012b5f949a6b662fdfc80b2c4bb23d3557d5"
    },
    {
      "path": "config.json",
      "bytes": 597,
      "sha256": "2f1e961e47c43f8378ea90c4c12cb0a21933eaa121070f0e29b004ddfd92f569"
    },
    {
      "path": "model.safetensors",
      "bytes": 123917994,
      "sha256": "e3998ec8e5f70ad5fa7682a1fa211b76d4adcb775f809b976b1a60506be113e6"
    }
  ],
  "totalBytes": 123940709,
  "referenceFiles": [
    {
      "path": "pytorch_model.bin",
      "bytes": 124072481,
      "sha256": "bb7df98fcb8576411cd97b34fdbb418bed22710fbf60943ba943fe46d332ff8b",
      "note": "pickle checkpoint of the same weights; not staged, not loaded; its Hub LFS SHA-256 is recorded for provenance only"
    }
  ]
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = MaxViTPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Download, verify and validate the labelled sample

`fetch_sample_archive` downloads `CIFAR-10-subset.zip` from the `Cleanlab/cifar-10-subset` dataset **at a pinned commit** and checks its byte size and SHA-256 before any member is opened. There is no fallback: a mismatch stops the notebook. `read_class_archive` then reads the `frog` and `truck` class folders. It refuses absolute member paths and `..` segments, and it bounds the member count and the uncompressed size before decompressing anything.

`validate_dataset` checks every record before any model runs: record keys, image type and size, and labels. It reports classes with too few images as errors, and imbalance and pixel-identical duplicates as findings.

**What to look for:** CIFAR-10 images are 32×32 pixels. The model's preprocessing upsamples them to 224×224, so every image the network sees here is a blurred thumbnail, far from the photographs ImageNet was collected from.

In [ ]:
import hashlib
import json
import os
from pathlib import Path

os.makedirs('outputs', exist_ok=True)
OUTPUTS = Path('outputs')
DATA_DIR = Path('data')

PER_CLASS = 200  # @param {type:"integer"}
DATASET_SEED = 0  # @param {type:"integer"}
EPOCHS = 5  # @param {type:"integer"}

archive_info = fetch_sample_archive(DATA_DIR, allow_download=True)
print({k: archive_info[k] for k in ('dataset_id', 'revision', 'license', 'bytes', 'sha256', 'fetched')})
records = read_class_archive(DATA_DIR / SAMPLE_DATASET_FILE, classes=SAMPLE_CLASSES, max_per_class=PER_CLASS, seed=DATASET_SEED)
dataset_manifest = validate_dataset(records, SAMPLE_CLASSES, epochs=EPOCHS)
print(json.dumps(dataset_manifest, indent=2))

preview = Image.new('RGB', (8 * 64, 2 * 64))
for row, label in enumerate(SAMPLE_CLASSES):
    for col, record in enumerate([r for r in records if r['label'] == label][:8]):
        preview.paste(record['image'].resize((64, 64), Image.NEAREST), (64 * col, 64 * row))
preview

## 5. Split, and measure two baselines before any training

The records are split per class with a fixed seed: 70% for training, and the rest halved into a **held-out** split, which scores every method below, and an **unseen** split, used only in Section 10. No hyperparameter is chosen on either. A random split is valid here because CIFAR-10 images are independent thumbnails; images that share a photograph or a camera session must be split by that group instead.

Two baselines give the fine-tune something to beat:

- **Majority class:** always answer the most frequent training label. On a balanced split this is 50%, and any useful model must clear it.
- **Zero-shot ImageNet mapping:** the pretrained 1000-class head already has frog and truck classes. `zero_shot_evaluate` sums the softmax mass over ImageNet's three frog classes and over its big-truck classes (fire engine, garbage truck, moving van, tow truck, trailer truck; CIFAR-10's `truck` excludes pickups), and answers whichever group is larger. No weight changes.

The cell also prints ImageNet top-5 predictions for four images and a `sample-sanity` evaluation report: does the true group appear in the top-1 or the top-5?

In [ ]:
SEED = 0  # @param {type:"integer"}
TOP_K = 5  # @param {type:"integer"}

train_records, rest = split_dataset(records, train_fraction=0.7, seed=SEED)
held_out, unseen = split_dataset(rest, train_fraction=0.5, seed=SEED)
ids = [{r['id'] for r in part} for part in (train_records, held_out, unseen)]
assert not (ids[0] & ids[1] or ids[0] & ids[2] or ids[1] & ids[2]), 'split leaked records'
TRAIN_MAJORITY = majority_class(train_records)
print({'train': len(train_records), 'held_out': len(held_out), 'unseen': len(unseen), 'train_majority': TRAIN_MAJORITY})

sample = held_out[:2] + held_out[-2:]
input_manifest = validate_inputs([r['image'] for r in sample], top_k=TOP_K, names=[r['id'] for r in sample])
try:
    validate_inputs([r['image'] for r in sample], top_k=0)
except ValueError as exc:
    input_manifest['findings'].append({'probe': 'top_k=0', 'rejected': str(exc)})
imagenet_result = pipe.predict([r['image'] for r in sample], top_k=TOP_K)
for record, pred in zip(sample, imagenet_result['predictions'], strict=True):
    print(record['label'], '->', [(item['label'].split(',')[0], round(item['score'], 3)) for item in pred['top_k']])
imagenet_report = evaluation_report(imagenet_result, [r['label'] for r in sample], groups=IMAGENET_GROUPS, sample_kind='sample')
print({'verdict': imagenet_report['verdict'], 'metrics': [(m['k'], m['value']) for m in imagenet_report['metrics']]})

zero_shot = pipe.zero_shot_evaluate(held_out, IMAGENET_GROUPS, majority=TRAIN_MAJORITY)
print(json.dumps({k: zero_shot[k] for k in ('accuracy', 'balanced_accuracy', 'majority_baseline_accuracy', 'mean_group_mass', 'per_class_recall')}, indent=2))

## 6. Degenerate input probes: blank canvas and noise

A closed-set classifier has no "none of these" answer: every image gets one of its classes, with a score that sums to 1 across them. The cell shows what the ImageNet head answers for a white image and for uniform noise, and how high its top score is.

**What to look for:** a confident label on structure-free input is a property of softmax classification, not evidence about the image. The same probe is repeated on the adapted two-class model in Section 10.

In [ ]:
degenerate = {}
for name, image in (('blank', blank_image()), ('noise', noise_image(0))):
    pred = pipe.predict(image, top_k=3)['predictions'][0]
    degenerate[name] = [(item['label'].split(',')[0], round(item['score'], 3)) for item in pred['top_k']]
print(json.dumps(degenerate, indent=2))

## 7. Replace the head and measure the untrained two-class model

`from_pretrained(class_names=SAMPLE_CLASSES)` loads the verified checkpoint again and replaces only the classification layer (`head.fc`, 512 features to 1000 classes) with a new 512-to-2 layer initialised under `SEED`. Every other weight, including the head's LayerNorm and Tanh layer, keeps its ImageNet value.

**Expect roughly chance.** A randomly initialised head carries no information about frogs or trucks. This row shows the floor, and that the fine-tune — not the re-heading — is what moves the score.

In [ ]:
adapter = MaxViTPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=SAMPLE_CLASSES, seed=SEED)
print({'class_names': list(adapter.class_names), 'reinitialised': list(adapter.reinitialised), 'device': adapter.device})
untrained = adapter.evaluate(held_out, majority=TRAIN_MAJORITY)
print({k: untrained[k] for k in ('accuracy', 'balanced_accuracy', 'per_class_recall')})

## 8. Bounded fine-tuning

This cell runs the real adaptation step in this runtime: gradient fine-tuning with cross-entropy over the two classes.

- **What trains:** with `FREEZE_BACKBONE = False` (the default) every weight trains. With `True`, only the head trains (`head.`: its LayerNorm, the Tanh layer and the new `fc`), and every BatchNorm layer in the MBConv blocks is held in evaluation mode, so the backbone keeps both its weights and its running statistics. The cell prints the trainable and total parameter counts.
- **Schedule:** `EPOCHS` epochs of AdamW at learning rate `1e-4` with weight decay `0.01`, batch size 16, float32, seed `SEED`, and no data augmentation: training images go through the same evaluation transform as test images.

**Read the loss as optimisation evidence only.** A falling training loss says the optimizer is fitting the training images; the held-out comparison in the next section is the task evidence.

In [ ]:
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 16  # @param {type:"integer"}
FREEZE_BACKBONE = False  # @param {type:"boolean"}

run = adapter.finetune(
    train_records,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    freeze_backbone=FREEZE_BACKBONE,
    progress=lambda row: print(f"epoch {row['epoch']}/{row['epochs']}  loss {row['loss']:.4f}"),
)
print(json.dumps({key: run[key] for key in ('freeze_backbone', 'trainable_parameters', 'total_parameters', 'epochs',
                                         'batch_size', 'learning_rate', 'optimizer', 'augmentation', 'precision', 'device')}, indent=2))

## 9. Evaluate on the held-out split

All four rows are scored on the same held-out images. `accuracy` is the fraction answered correctly; `balanced_accuracy` averages the per-class recall, so it cannot be inflated by favouring the larger class. The confusion matrix shows which way the errors go. These are tutorial metrics from one pass over a few dozen thumbnails, with no dispersion estimate: a difference of one or two images is within noise.

In [ ]:
adapted = adapter.evaluate(held_out, majority=TRAIN_MAJORITY)
comparison = {
    'majority class': {'accuracy': adapted['majority_baseline_accuracy'], 'balanced_accuracy': 1 / len(SAMPLE_CLASSES)},
    'zero-shot ImageNet mapping': zero_shot,
    'untrained two-class head': untrained,
    'fine-tuned': adapted,
}
print(f"{'method':<28s} {'accuracy':>9s} {'balanced':>9s}")
for name, row in comparison.items():
    print(f"{name:<28s} {row['accuracy']:>9.3f} {row['balanced_accuracy']:>9.3f}")
print('confusion (rows true, columns predicted):', adapted['confusion_matrix'])

## 10. Inference on the unseen split and the degenerate probes again

The unseen split was never used for training or for any comparison above. The adapted pipeline classifies it, and the cell repeats the blank and noise probes: the two-class model must now answer `frog` or `truck` for them, whatever they contain.

In [ ]:
unseen_metrics = adapter.evaluate(unseen, majority=TRAIN_MAJORITY)
print({k: unseen_metrics[k] for k in ('accuracy', 'balanced_accuracy', 'n')})
unseen_rows = []
for record, pred in zip(unseen, adapter.predict([r['image'] for r in unseen[:MAX_BATCH]], top_k=2)['predictions'], strict=False):
    unseen_rows.append({'id': record['id'], 'truth': record['label'], 'predicted': pred['predicted_label'], 'score': round(pred['top_k'][0]['score'], 4)})
print(json.dumps(unseen_rows[:6], indent=2))
adapted_degenerate = {name: adapter.predict(image, top_k=2)['predictions'][0]['top_k'] for name, image in (('blank', blank_image()), ('noise', noise_image(0)))}
print({name: [(i['label'], round(i['score'], 3)) for i in top] for name, top in adapted_degenerate.items()})

## 11. Adapter export, fresh reload, and equivalence check

`save_artifact` writes `outputs/maxvit_adapter.safetensors`: every tensor the fine-tune could change, plus a metadata header naming the base model, its pinned revision, the base `model.safetensors` SHA-256, the class names and the frozen prefixes. With the default full fine-tune that is the whole state dict; with `FREEZE_BACKBONE = True` it is the head only.

`load_artifact` then builds a **fresh** pipeline from the verified base snapshot, loads the adapter tensors onto it, and refuses an adapter whose format, base identity, base digest or tensor set does not fit. The cell compares the reloaded predictions with the in-memory model's on the unseen split, with a stated tolerance: loading succeeding is not the check, reproducing the predictions is.

In [ ]:
artifact_path = OUTPUTS / 'maxvit_adapter.safetensors'
descriptor = adapter.save_artifact(artifact_path, notes='MaxViT-Tiny CIFAR-10 frog/truck tutorial adapter')
print(json.dumps(descriptor, indent=2))

reloaded = MaxViTPipeline.load_artifact(artifact_path, weights_dir=WEIGHTS_DIR)
print({'reloaded_source': reloaded.source, 'adapted': reloaded.adapted, 'class_names': list(reloaded.class_names)})

TOLERANCE = 1e-4
check_images = [r['image'] for r in unseen[:MAX_BATCH]]
first = adapter.predict(check_images, top_k=2)['predictions']
second = reloaded.predict(check_images, top_k=2)['predictions']
for a, b in zip(first, second, strict=True):
    assert a['predicted_label'] == b['predicted_label']
    assert abs(a['top_k'][0]['score'] - b['top_k'][0]['score']) <= TOLERANCE
reload_check = {'images_compared': len(check_images), 'tolerance': TOLERANCE, 'equivalent': True}
print(reload_check)

## 12. Write machine-readable outputs and provenance

The cell writes:
- `outputs/maxvit_classification_input_manifest.json`
- `outputs/maxvit_classification_evaluation_report.json`
- `outputs/maxvit_classification_result.json` (identity, runtime versions, device, dataset provenance and manifest, split, baselines, fine-tuning configuration, held-out and unseen metrics, adapter descriptor and reload check)
- `outputs/maxvit_classification_predictions.csv`
- `outputs/maxvit_adapter.safetensors` (written in Section 11)

In [ ]:
import csv

with open(OUTPUTS / 'maxvit_classification_input_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(input_manifest, f, indent=2)

with open(OUTPUTS / 'maxvit_classification_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(imagenet_report, f, indent=2)

with open(OUTPUTS / 'maxvit_classification_predictions.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['split', 'id', 'truth', 'predicted', 'score'])
    for split_name, part in (('held_out', held_out), ('unseen', unseen)):
        for start in range(0, len(part), MAX_BATCH):
            chunk = part[start:start + MAX_BATCH]
            for record, pred in zip(chunk, adapter.predict([r['image'] for r in chunk], top_k=1)['predictions'], strict=True):
                writer.writerow([split_name, record['id'], record['label'], pred['predicted_label'], f"{pred['top_k'][0]['score']:.4f}"])

result_export = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__,
                'timm': timm.__version__, 'cuda': torch.cuda.is_available()},
    'device': pipe.device,
    'dataset': {'archive': archive_info, 'manifest': dataset_manifest, 'per_class': PER_CLASS, 'seed': DATASET_SEED},
    'split': {'train': len(train_records), 'held_out': len(held_out), 'unseen': len(unseen), 'seed': SEED, 'train_majority': TRAIN_MAJORITY},
    'imagenet_top_k': imagenet_result['predictions'],
    'degenerate_probes': {'imagenet_head': degenerate, 'adapted': adapted_degenerate},
    'baselines': {'zero_shot': zero_shot, 'untrained_head': untrained},
    'finetune': run,
    'held_out': adapted,
    'unseen': unseen_metrics,
    'artifact': descriptor,
    'reload_check': reload_check,
}
with open(OUTPUTS / 'maxvit_classification_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_export, f, indent=2, default=str)

for p in sorted(OUTPUTS.iterdir()):
    if p.is_file():
        print(f'  {p.name:<48s} {p.stat().st_size:>12,d} bytes')

## 13. Optional: Bring Your Own Data (BYOD)

Both branches are off by default, so `Run all` never stops here. Before you turn one on, read the contract:

- **Image branch** (`USE_BYOD_IMAGE`): one image file PIL can open, each side between `MIN_IMAGE_SIDE` (8) and `MAX_IMAGE_SIDE` (4096) px. It runs through `validate_inputs`, the ImageNet head's `predict` and `evaluation_report`; the report is `not-measurable`, because no label comes with the image.
- **Dataset branch** (`USE_BYOD_DATASET`): a directory or a `.zip` of class folders, `<class>/<image>`, with at least 2 classes and at least 2 images per class, at most 5,000 images. Archive members must stay inside the archive; folder files must not link outside the directory. The class names are the folder names. The branch runs the same validate → split → baselines → fine-tune → evaluate → export → reload stages as the sample; the zero-shot ImageNet baseline needs a class-to-ImageNet mapping, so it is skipped here.

Set `BYOD_IMAGE_PATH` or `BYOD_DATASET_PATH` to read from a mounted or local location; leave them empty on Colab to get an upload dialog instead. Uploaded files are written under `outputs/byod/` in this runtime and are not sent anywhere else. The first lines of the cell show the validator refusing two malformed inputs with messages that name the failed rule.

In [ ]:
USE_BYOD_IMAGE = False  # @param {type:"boolean"}
BYOD_IMAGE_PATH = ""  # @param {type:"string"}
USE_BYOD_DATASET = False  # @param {type:"boolean"}
BYOD_DATASET_PATH = ""  # @param {type:"string"}

for desc, probe in (
    ('non-image object', lambda: validate_inputs('/not/an/image.png')),
    ('class with one image', lambda: validate_dataset([{'image': blank_image(32, 32), 'label': 'frog'}, {'image': noise_image(1, 32, 32), 'label': 'truck'}], SAMPLE_CLASSES)),
):
    try:
        probe()
    except (TypeError, ValueError) as exc:
        print(f'refused as expected: {desc} -> {type(exc).__name__}: {exc}')

BYOD_DIR = OUTPUTS / 'byod'

def _upload_into(target):
    from google.colab import files  # type: ignore[import-not-found]
    target.mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():
        (target / Path(name).name).write_bytes(data)
    return target

if USE_BYOD_IMAGE:
    image_path = Path(BYOD_IMAGE_PATH) if BYOD_IMAGE_PATH else next(iter(sorted(_upload_into(BYOD_DIR / 'image').iterdir())))
    with Image.open(image_path) as handle:
        byod_image = handle.convert('RGB')
    print(validate_inputs(byod_image, top_k=TOP_K, names=[image_path.name])['verdict'])
    byod_result = pipe.predict(byod_image, top_k=TOP_K)
    print([(item['label'].split(',')[0], round(item['score'], 3)) for item in byod_result['predictions'][0]['top_k']])
    print(evaluation_report(byod_result, None, sample_kind='byod')['verdict'])
else:
    print('BYOD image branch is off; set USE_BYOD_IMAGE = True to classify your own image.')

if USE_BYOD_DATASET:
    source = Path(BYOD_DATASET_PATH) if BYOD_DATASET_PATH else next(iter(sorted(_upload_into(BYOD_DIR / 'dataset').iterdir())))
    byod_records = read_class_archive(source) if source.suffix.lower() == '.zip' else read_class_folder(source)
    byod_names = sorted({r['label'] for r in byod_records})
    print(json.dumps(validate_dataset(byod_records, byod_names, epochs=EPOCHS), indent=2))
    byod_train, byod_held = split_dataset(byod_records, train_fraction=0.7, seed=SEED)
    byod_majority = majority_class(byod_train)
    byod_pipe = MaxViTPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=byod_names, seed=SEED)
    byod_before = byod_pipe.evaluate(byod_held, majority=byod_majority)
    byod_pipe.finetune(byod_train, epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, seed=SEED, freeze_backbone=FREEZE_BACKBONE)
    byod_after = byod_pipe.evaluate(byod_held, majority=byod_majority)
    print({'majority_baseline': byod_after['majority_baseline_accuracy'], 'untrained': byod_before['balanced_accuracy'], 'fine_tuned': byod_after['balanced_accuracy']})
    byod_descriptor = byod_pipe.save_artifact(OUTPUTS / 'byod_maxvit_adapter.safetensors', notes='BYOD adaptation adapter')
    MaxViTPipeline.load_artifact(OUTPUTS / 'byod_maxvit_adapter.safetensors', weights_dir=WEIGHTS_DIR)
    print('BYOD adapter exported and reloaded:', byod_descriptor['sha256'][:16])
else:
    print('BYOD dataset branch is off; set USE_BYOD_DATASET = True to adapt MaxViT-Tiny on your own class folders.')

## Interpretation and limits

**What this notebook established, in this runtime.** The pinned `timm/maxvit_tiny_tf_224.in1k` snapshot was verified against a committed SHA-256 manifest before loading, and the pinned CIFAR-10 subset was verified against its recorded digest before it was read. The ImageNet head was run on sample thumbnails and on two structure-free probes, and its frog and truck classes gave a zero-shot baseline. The head was then replaced for the two tutorial classes and fine-tuned, and the result was compared with the majority-class, zero-shot and untrained baselines on the same held-out split and checked on an unseen split. The adapter was exported, reloaded onto a fresh base model and checked against the in-memory predictions.

**What a green run proves.** Successful execution proves that the recorded repository revision, the pinned dependency set, the pinned checkpoint and the pinned dataset together reproduce these stages in a fresh runtime, without the repository being cloned or installed and without any DIMER worker or service. It does **not** establish benchmark superiority, fitness for any deployment, or that the adapted model recognises frogs or trucks outside CIFAR-10's 32×32 thumbnails. The held-out scores come from a few dozen images and carry no dispersion estimate. The scores are not calibrated probabilities, and there is no reject option.

**Reproducibility.** Seeds are form fields (`DATASET_SEED`, `SEED`), the run is float32 with no data augmentation, and the new head is initialised under `SEED`. GPU kernels are not forced to be deterministic, so repeated GPU runs can differ in the last digits of the loss and the scores.

**Try next.** Set `FREEZE_BACKBONE = True` and compare the held-out scores and runtime of a head-only fine-tune, or lower `PER_CLASS` and watch how quickly the fine-tuned model falls back toward the zero-shot baseline. To transfer the workflow, point `BYOD_DATASET_PATH` at a small set of class folders from your own domain.

## References

- Tu, Z., Talebi, H., Zhang, H., Yang, F., Milanfar, P., Bovik, A. and Li, Y. (2022). *MaxViT: Multi-Axis Vision Transformer.* ECCV 2022. [arXiv:2204.01697](https://arxiv.org/abs/2204.01697).
- Hugging Face checkpoint: [timm/maxvit_tiny_tf_224.in1k](https://huggingface.co/timm/maxvit_tiny_tf_224.in1k) — Apache-2.0; upstream code and weights: [google-research/maxvit](https://github.com/google-research/maxvit) — Apache-2.0; port: [huggingface/pytorch-image-models](https://github.com/huggingface/pytorch-image-models).
- Krizhevsky, A. (2009). *Learning Multiple Layers of Features from Tiny Images.* Technical report, University of Toronto — the CIFAR-10 dataset.
- Sample archive: [Cleanlab/cifar-10-subset](https://huggingface.co/datasets/Cleanlab/cifar-10-subset) — MIT licence.
- Repository model card: https://github.com/kurtvalcorza/maxvit-classification-pipeline/blob/main/MODEL_CARD.md
- [`kurtvalcorza/maxvit-classification-pipeline`](https://github.com/kurtvalcorza/maxvit-classification-pipeline) — source repository for this pipeline.